# Visualizing the Author-Interaction Networks — the Social-Media Map

Renders the project's **directed, weighted author-interaction networks** (node =
author, edge weighted by retweets/replies/quotes) as *social-media maps*: GPU
**ForceAtlas2** layouts (the algorithm behind the classic Gephi Twitter maps)
with **community-colored ink**, in **light and dark** variants.

**Pipeline in one line:** load the `.gml` with igraph → undirected backbone →
precomputed Leiden communities → `cugraph.force_atlas2` on a **Colab GPU**
(two recipes: `classic` = separated community continents, `linlog` = soft
radial nebula) → datashader render with community-colored straight edges.

**Requirements:** a **GPU runtime** (Runtime → Change runtime type → GPU; any of
T4/L4/A100). The 2M-node layout takes ~10–16 s on an A100.

**Where the exploration went:** every alternative tried on the way here (CPU
grid-FR, DrL's failure at scale, k-core reduction, percentile reframing, hammer
bundling, the FA2 parameter sweep) is codified in the knowledge base —
`content/how-to/NETWORK_VISUALIZATION_SKILL.md` (how to choose) and
`content/reference/NETWORK_VIZ_STYLES_REF.md` (every style, with code, light &
dark). This notebook keeps only the production path.

> **How to use:** set `NETWORK` in the Setup cell, connect a GPU runtime, then
> *Runtime → Run all*. If the selected `.gml` is missing, a synthetic network
> is generated so every cell still runs end-to-end.


In [ ]:
import os
from pathlib import Path
import sys

# --- ENVIRONMENT SWITCH ---
# Set to True if running on local machine with Google Drive Desktop mounted
# Set to False if running in Google Colab cloud
RUNNING_LOCALLY = False

if RUNNING_LOCALLY:
    # --- REPO ROOT ON sys.path (harmless here; this notebook uses no src imports) ---
    _REPO_ROOT = str(Path(os.getcwd()).resolve().parents[1])
    if _REPO_ROOT not in sys.path:
        sys.path.insert(0, _REPO_ROOT)
    # Standard macOS path for Google Drive Desktop
    BASE_PATH = Path('/Volumes/GoogleDrive/My Drive/Colab Projects/AI Public Trust')
else:
    # Google Colab cloud path
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/My Drive/Colab Projects/AI Public Trust')

# Shared folders (same variables as the other notebooks)
datasets_folder = BASE_PATH / 'Data Sets'
networks_folder = BASE_PATH / 'Data Sets/Networks/'

# --- WHICH NETWORK TO VISUALIZE --------------------------------------------
# Registry of every network written by the generation / analysis notebooks.
# All are directed, weighted author-interaction graphs in GML, so the same
# pipeline renders any of them. Each entry: key -> (filename, source_nb, note).
NETWORKS = {
    'Test':                    ('Test_Network.gml',                          '02', 'small test interaction network'),
    'Full':                    ('Full_Network.gml',                          '02', 'full author-interaction network (large)'),
    'LWCC':                    ('LWCC.gml',                                   '01', 'largest weakly-connected component of Full'),
    '90TS_LWCC':               ('90TS_LWCC.gml',                             '01', 'total-strength pruned (keep 90% of weight), then LWCC'),
    'Final_OutThreshold1':     ('Final_OutThreshold1.gml',                   '01', 'out-strength >= 1.0 pruned network'),
    'Final_leiden_fast':       ('Final_OutThreshold1_leiden_fast.gml',       '01', 'out-thresh network + Leiden-fast community labels'),
    'Final_louvain':           ('Final_OutThreshold1_louvain.gml',           '01', 'out-thresh network + Louvain community labels'),
    'Final_label_propagation': ('Final_OutThreshold1_label_propagation.gml', '01', 'out-thresh network + label-propagation community labels'),
}

# <-- pick any key from NETWORKS above
NETWORK = 'Final_leiden_fast'

_fname, _src, _note = NETWORKS[NETWORK]
GML_PATH = networks_folder / _fname
# Visualization artifacts land next to the networks on Drive, one folder per network.
OUTPUT_DIR = networks_folder / f'viz_outputs_{NETWORK}'

print(f'NETWORK={NETWORK!r}  (from notebook {_src}: {_note})')
print(f'  GML_PATH   = {GML_PATH}')
print(f'  OUTPUT_DIR = {OUTPUT_DIR}')

## 0. Setup

### 0.1 Install dependencies

Colab ships `networkx`, `matplotlib`, `pandas`, `numpy`, `scipy`. We add the
graph engine (`python-igraph`) and the renderer (`datashader` + `colorcet`),
plus small utilities. The GPU layout engine (`cugraph`) is installed in §4
with a version pin. Guarded by `if not RUNNING_LOCALLY` so it is a no-op
locally.


In [ ]:
import os
if not RUNNING_LOCALLY:
    print('Running Colab setup shell commands...')
    !pip install -q python-igraph datashader colorcet ipysigma "dask[dataframe]" scikit-image pyarrow tqdm psutil
else:
    print('Running locally: Skipping Colab shell setup.')


### 0.2 Imports

All third-party imports the notebook uses, listed explicitly (per
`notebook_setup.md`). `ipysigma`'s inline widget needs Colab's custom widget
manager; the `try/except` keeps the notebook runnable in plain Jupyter/VS Code.


In [ ]:
from pathlib import Path
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil                              # memory guardrails
from tqdm.auto import tqdm                 # progress bars
import networkx as nx                      # used as interchange format only
import igraph as ig                        # the workhorse: I/O, communities, layout

import datashader as ds                    # large-scale rasterization
import datashader.transfer_functions as tf
from datashader.bundling import connect_edges, hammer_bundle
import colorcet as cc                      # perceptually-designed palettes (glasbey)

# ipysigma needs Colab's custom widget manager to display inline widgets.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"igraph {ig.__version__} | datashader {ds.__version__} | "
      f"networkx {nx.__version__} | colab: {IN_COLAB}")


### 0.3 Configuration

Every knob you might want to touch lives in this one cell. (`GML_PATH` and
`OUTPUT_DIR` were set by the environment cell from the `NETWORK` switch.)


In [ ]:
# ----------------------------- CONFIGURATION ------------------------------ #
RANDOM_SEED = 42                           # reproducible layouts & sampling

# --- Communities ---
# The Final_* networks carry a precomputed 'community_<method>' vertex
# attribute. When True, color by those labels; else run Leiden below.
USE_PRECOMPUTED_COMMUNITY = True

# --- Layout: GPU ForceAtlas2 recipes (kb: NETWORK_VIZ_STYLES_REF.md) --------
# 'classic' -> separated community continents (the social-media map look);
# 'linlog'  -> soft radial nebula (needs its own scaling sweep for separation).
FA2_RECIPES = {
    "classic": dict(lin_log_mode=False, outbound_attraction_distribution=True,
                    strong_gravity_mode=False, gravity=1.0, scaling_ratio=2.0),
    "linlog":  dict(lin_log_mode=True,  outbound_attraction_distribution=True,
                    strong_gravity_mode=False, gravity=1.0, scaling_ratio=2.0),
}
FA2_ITER = 600

# Reframe: clip node coords to this central percentile per axis before scaling
# to [0,1], so a few outliers can't define the frame (100.0 = no clip).
CLIP_PCT = 99.0

# --- Rendering ---
CANVAS_PX = 1600                           # raster size in pixels (square)
TOP_COMMUNITIES = 16                       # communities that get their own color
DARK_BG = "#0d0d14"                        # near-black, slightly blue
EDGE_ALPHA = {"light": 70, "dark": 150}    # edges recede on white, glow on dark
NODE_MIN_ALPHA = {                         # log-shading alpha floor = the color-
    "classic": {"light": 90,  "dark": 110},  # saliency lever; the diffuse linlog
    "linlog":  {"light": 140, "dark": 160},  # cloud needs a higher floor
}

# --- Artifacts ---
EXPORT_GRAPHML = False                     # ~1.2 GB at 2M nodes; enable if needed

# --- Guardrails: per-stage time budgets (seconds) ---
TIME_BUDGETS = {"layout": 900, "bundling": 900, "render": 300}
ABORT_IF_ESTIMATE_EXCEEDS_BUDGET = True

# --------------------------------------------------------------------------- #
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", category=FutureWarning)
print("config loaded | FA2 recipes:", list(FA2_RECIPES), "| output dir:", OUTPUT_DIR)

### 0.4 Progress bars & guardrails

Long cells that print nothing are indistinguishable from hung cells. This
toolkit fixes that for every expensive stage:

* **`Stage`** — announces each step, times it, logs peak memory (§8 prints a
  summary table).
* **`run_guarded`** — the important one. igraph's layouts are C calls that
  **hold the GIL**: a progress thread can't tick during them, and `Ctrl-C`
  won't interrupt them. So we run them in a **forked subprocess**, show a live
  elapsed ticker in the notebook, and **terminate them** if they blow the time
  budget. Nothing can hang forever.
* **`require`** — hard assertions with actionable messages, so a bad layout
  fails *here* rather than silently producing an empty PNG three cells later.

> *Why no per-iteration bar for the layout?* Feeding igraph's FR intermediate
> coordinates (`seed=…`) to chunk it disables its grid optimization — measured
> **~160× slower** and lower quality. An elapsed ticker + ETA + kill switch is
> strictly better than a "pretty" bar that costs you an hour.

In [ ]:
import multiprocessing as mp
from contextlib import contextmanager

STAGE_LOG = []                                   # filled by Stage, printed in §8


def require(condition: bool, message: str) -> None:
    """Hard guardrail: abort the run with an actionable message."""
    if not condition:
        raise RuntimeError(f"GUARDRAIL FAILED - {message}")


def check_memory(need_gb: float, what: str) -> None:
    """Refuse to start a stage that obviously won't fit in RAM."""
    avail = psutil.virtual_memory().available / 1e9
    require(need_gb < avail,
            f"{what} needs ~{need_gb:.1f} GB but only {avail:.1f} GB is free. "
            f"Reduce the graph (k-core / top-degree) or the parameters.")
    if need_gb > 0.6 * avail:
        print(f"  warning: {what} may use ~{need_gb:.1f} GB of {avail:.1f} GB free")


@contextmanager
def Stage(name: str):
    """Time a stage, report it, and record it for the final summary table."""
    print(f"> {name} ...", flush=True)
    t0 = time.time()
    ok = True
    try:
        yield
    except BaseException:
        ok = False
        raise                                    # never swallow the error
    finally:
        dt = time.time() - t0
        rss = psutil.Process().memory_info().rss / 1e9
        STAGE_LOG.append({"stage": name, "seconds": round(dt, 1),
                          "status": "ok" if ok else "FAILED", "RSS_GB": round(rss, 2)})
        print(f"{'v' if ok else 'x'} {name}: {dt:.1f}s (RSS {rss:.1f} GB)", flush=True)


def _worker(conn, fn, args, kwargs):
    """Child-process entry point: run fn, ship back the result or the error."""
    try:
        conn.send(("ok", fn(*args, **kwargs)))
    except BaseException as e:                   # noqa: BLE001
        conn.send(("err", repr(e)))
    finally:
        conn.close()


def run_guarded(fn, *args, desc: str, timeout_s: float, eta_s: float = None,
                tick: float = 1.0, **kwargs):
    """Run fn in a forked subprocess with a live ticker and a hard kill switch.

    Used for stages that would otherwise be black boxes:
      - igraph layouts: C code that holds the GIL (uninterruptible in-process)
      - hammer_bundle: long, and worth being able to abort

    Guarantees: you always see elapsed time; the stage cannot run past
    `timeout_s`; a child killed by the OS (OOM) is reported as such.
    """
    ctx = mp.get_context("fork")                 # fork => no pickling of inputs
    parent, child = ctx.Pipe(duplex=False)
    proc = ctx.Process(target=_worker, args=(child, fn, args, kwargs), daemon=True)
    t0 = time.time()
    proc.start()
    child.close()                                # only the child holds the writer

    eta = f" / est {eta_s:.0f}s" if eta_s else ""
    fmt = "{desc} | elapsed {elapsed}" + eta + f" | budget {timeout_s:.0f}s"
    try:
        with tqdm(desc=desc, bar_format=fmt) as bar:
            while not parent.poll(tick):         # poll = the heartbeat
                bar.refresh()                    # keeps the ticker alive
                if not proc.is_alive():          # died without sending anything
                    require(False, f"'{desc}' subprocess died (exit "
                                   f"{proc.exitcode}) - almost always out-of-memory")
                if time.time() - t0 > timeout_s:
                    proc.terminate()
                    require(False, f"'{desc}' exceeded its {timeout_s:.0f}s budget "
                                   f"and was killed. Raise TIME_BUDGETS, shrink the "
                                   f"graph, or switch to a cheaper method.")
        status, payload = parent.recv()
    finally:
        if proc.is_alive():
            proc.terminate()
        proc.join()

    require(status == "ok", f"'{desc}' failed inside the subprocess: {payload}")
    return payload


def validate_positions(xy: np.ndarray, n_expected: int) -> None:
    """Catch the three ways a layout silently goes wrong."""
    require(xy.shape == (n_expected, 2),
            f"layout returned {xy.shape}, expected ({n_expected}, 2)")
    require(np.isfinite(xy).all(),
            "layout contains NaN/inf - usually caused by non-positive edge weights")
    require(float((xy.max(0) - xy.min(0)).max()) > 0,
            "layout is degenerate (every node at the same point)")


print("guardrails armed | budgets:", TIME_BUDGETS)

## 1. Load the network

### 1.1 A robust GML loader

GML is a loosely specified format, and files written by Gephi, NetworkX,
igraph or hand-rolled exporters differ in small ways. The strategy below is:
**try igraph's C parser first** (orders of magnitude faster and far more
memory-efficient on big files), and only fall back to `networkx.read_gml`
(slower, but very tolerant) if it fails — converting the result back to
igraph so the rest of the pipeline is identical either way.

In [ ]:
def load_gml(path: Path) -> ig.Graph:
    """Read a .gml file into an igraph Graph, trying the fast parser first.

    Returns an igraph.Graph. Vertex attribute ``node_label`` is guaranteed to
    exist (best-effort human-readable name), and ``weight`` on edges is
    coerced to float when present.
    """
    path = Path(path)
    t0 = time.time()
    try:
        g = ig.Graph.Read_GML(str(path))                 # C parser: fast path
        parser = "igraph"
    except Exception as err:                             # noqa: BLE001 - GML is messy
        print(f"igraph parser failed ({err!r}); falling back to networkx …")
        gx = nx.read_gml(path, label="id")               # label='id' avoids
        g = ig.Graph.from_networkx(gx)                   # duplicate-label errors
        parser = "networkx"
    print(f"Parsed with {parser} in {time.time() - t0:.1f}s")

    # ---- Normalize a display name onto every vertex -------------------- #
    # GML files may carry 'label', 'name', 'id', or nothing at all.
    vattrs = g.vs.attributes()
    for candidate in ("label", "name", "_nx_name", "id"):
        if candidate in vattrs:
            g.vs["node_label"] = [str(v) for v in g.vs[candidate]]
            break
    else:
        g.vs["node_label"] = [str(i) for i in range(g.vcount())]

    # ---- Make sure edge weights (if any) are usable floats ------------- #
    if "weight" in g.es.attributes():
        w = np.asarray(g.es["weight"], dtype=float)
        w[~np.isfinite(w)] = 1.0
        # force-directed layouts require strictly positive weights
        w[w <= 0] = w[w > 0].min() if (w > 0).any() else 1.0
        g.es["weight"] = w.tolist()
    return g

### 1.2 Demo fallback: a synthetic social-media-like network

If `GML_PATH` doesn't exist we build a stochastic block model (~30 k nodes,
~40 unequal communities, weighted edges) and write it to disk, so the rest of
the notebook demonstrates itself. **Delete nothing — this cell is a no-op
when your real file is present.**

In [ ]:
def make_demo_network(path: Path, n_blocks: int = 40, seed: int = RANDOM_SEED) -> None:
    """Write a synthetic weighted 'social network' to *path* as GML.

    Structure = SBM (communities) + a power-law overlay (hubs/influencers),
    which reproduces the two signatures of real platforms: modular structure
    AND a heavy-tailed degree distribution.
    """
    rng = np.random.default_rng(seed)

    # Unequal community sizes (lognormal) look far more like real platforms
    # than equal blocks do.
    sizes = np.maximum(80, rng.lognormal(mean=6.3, sigma=0.55, size=n_blocks).astype(int))
    n = int(sizes.sum())

    # Denser inside communities than between them. p_in is scaled per block so
    # every community has a similar expected internal degree.
    p_out = 2.0 / n
    p_in = np.minimum(0.9, 12.0 / sizes)              # ~12 internal neighbors each
    pref = np.full((n_blocks, n_blocks), p_out)
    np.fill_diagonal(pref, p_in)

    try:    # igraph >= 1.0 signature
        sbm = ig.Graph.SBM(pref.tolist(), sizes.tolist())
    except TypeError:  # igraph 0.x signature had a leading n argument
        sbm = ig.Graph.SBM(n, pref.tolist(), sizes.tolist())

    # Hub overlay: a static power-law graph on the same vertices adds
    # celebrity-style high-degree nodes (exponent 2.0 -> very heavy tail).
    hubs = ig.Graph.Static_Power_Law(n, 2 * n, exponent_out=2.0)
    g = ig.Graph(n=n, edges=sbm.get_edgelist() + hubs.get_edgelist()).simplify()

    g.vs["label"] = [f"user_{i:05d}" for i in range(n)]
    # Heavy-tailed interaction weights, like retweet/reply counts.
    g.es["weight"] = np.round(rng.pareto(2.5, g.ecount()) + 1, 2).tolist()
    g.write_gml(str(path))
    print(f"Demo network written to {path}: {g.vcount():,} nodes / {g.ecount():,} edges")


if not GML_PATH.exists():
    # IMPORTANT: never write the synthetic graph into networks_folder under a
    # real network's filename - a later analysis run could mistake it for real
    # data. The demo lives in OUTPUT_DIR and GML_PATH is repointed in-session.
    demo_path = OUTPUT_DIR / "demo_network.gml"
    print(f"'{GML_PATH}' not found - generating a demo network at '{demo_path}' instead.")
    if not demo_path.exists():
        make_demo_network(demo_path)
    GML_PATH = demo_path

with Stage("load GML"):
    G_raw = load_gml(GML_PATH)

# Guardrails: fail now, not five cells later.
require(G_raw.vcount() > 0, f"{GML_PATH} parsed to an empty graph")
require(G_raw.ecount() > 0, f"{GML_PATH} has nodes but no edges - nothing to lay out")
check_memory(G_raw.ecount() * 250 / 1e9, "layout + rendering tables")

### 1.3 First look

Before drawing anything, know what you're holding: order, size, density,
degree statistics and how fragmented the graph is. These numbers also drive
choices later (layout method, whether the interactive view needs reducing).

In [ ]:
def describe(g: ig.Graph, name: str = "graph") -> None:
    """Print a compact structural summary of an igraph Graph."""
    deg = np.asarray(g.degree())
    comps = g.connected_components(mode="weak")
    pairs = max(1, g.vcount() * (g.vcount() - 1))       # ordered vertex pairs
    density = g.ecount() / pairs * (1 if g.is_directed() else 2)
    print(
        f"--- {name} ---\n"
        f"nodes: {g.vcount():>12,}\n"
        f"edges: {g.ecount():>12,}\n"
        f"directed: {g.is_directed()},  weighted: {'weight' in g.es.attributes()}\n"
        f"density: {density:.2e}\n"
        f"degree  mean/median/max: {deg.mean():.1f} / {np.median(deg):.0f} / {deg.max():,}\n"
        f"components: {len(comps):,} (largest: {max(comps.sizes()):,} nodes)"
    )

describe(G_raw, "raw network")

## 2. Preprocess for visualization

Three standard steps before spatializing a big social graph. None of them
change your *data on disk* — they produce the view we draw.

### 2.1 Undirect, deduplicate, drop self-loops

Force-directed layouts and most community detection operate on the undirected
backbone. Parallel edges are collapsed (summing weights, so a pair that
interacted 10× still pulls 10× harder) and self-loops — meaningless for
spatialization — are dropped.

In [ ]:
def to_viz_backbone(g: ig.Graph) -> ig.Graph:
    """Undirected, simple (no loops / multi-edges) copy for layout & rendering."""
    h = g.copy()
    has_w = "weight" in h.es.attributes()
    if h.is_directed():
        # collapse: a<->b reciprocal pairs become one edge; weights are summed
        h = h.as_undirected(mode="collapse",
                            combine_edges={"weight": "sum"} if has_w else None)
    h = h.simplify(multiple=True, loops=True,
                   combine_edges={"weight": "sum"} if has_w else None)
    return h

G = to_viz_backbone(G_raw)
describe(G, "undirected simple backbone")

### 2.2 Keep the giant component

Force-directed layouts scatter disconnected fragments arbitrarily around the
frame, burying the structure you care about. The near-universal convention is
to lay out and draw the **giant (largest) connected component** and report
what was dropped. If your analysis needs the fragments, visualize them
separately.

In [ ]:
components = G.connected_components(mode="weak")
G_giant = components.giant()

dropped_nodes = G.vcount() - G_giant.vcount()
print(f"Keeping giant component: {G_giant.vcount():,} nodes / {G_giant.ecount():,} edges "
      f"({dropped_nodes:,} nodes in {len(components) - 1:,} smaller components dropped)")

### 2.3 Optional reducers (kept as tools, not applied)

For *extremely* large or noisy graphs two standard reductions help before the
interactive view — we define them here and use the degree-based one later in
§5.4:

* **k-core**: iteratively strip nodes of degree < k; removes the periphery
  fuzz while preserving the dense heart of the network.
* **top-degree induced subgraph**: keep the N highest-degree accounts and the
  edges among them — the "who matters" view.

In [ ]:
def k_core(g: ig.Graph, k: int) -> ig.Graph:
    """Return the k-core of g (max subgraph where every node has degree >= k)."""
    coreness = np.asarray(g.coreness())          # shell index per vertex, O(E)
    return g.induced_subgraph(np.flatnonzero(coreness >= k).tolist())

def top_degree_subgraph(g: ig.Graph, n_keep: int) -> ig.Graph:
    """Induced subgraph on the n_keep highest-degree vertices (then its giant CC)."""
    if g.vcount() <= n_keep:
        return g
    order = np.argsort(g.degree())[::-1][:n_keep]
    sub = g.induced_subgraph(order.tolist())
    return sub.connected_components(mode="weak").giant()

# Kept as TOOLS, not applied: FA2's outbound attraction handles the ~1.27M
# degree-1 leaves gracefully, so the full graph renders well. Set KCORE_K = 2
# to restrict the map to the "conversation core" of repeat participants.
KCORE_K = None
if KCORE_K:
    _before = G_giant.vcount()
    G_giant = k_core(G_giant, KCORE_K)
    print(f"applied {KCORE_K}-core: {G_giant.vcount():,} nodes "
          f"(dropped {_before - G_giant.vcount():,} of {_before:,})")
else:
    print(f"2-core would keep {k_core(G_giant, 2).vcount():,} of "
          f"{G_giant.vcount():,} nodes  [not applied - full graph]")

## 3. Compute the structure we will encode visually

A hairball with uniform dots says nothing. The two attributes that carry the
most information in social-network figures are **degree** (→ node size) and
**community** (→ node color).

### 3.1 Degree

In [ ]:
degree = np.asarray(G_giant.degree())
weights = "weight" if "weight" in G_giant.es.attributes() else None
print(f"degree: mean {degree.mean():.1f}, p90 {np.percentile(degree, 90):.0f}, "
      f"max {degree.max():,}")

### 3.2 Communities

Node color encodes community. Two sources, in order of preference:

1. **Precomputed labels** — the `Final_OutThreshold1_*` networks from
   `01_network_analysis.ipynb` already store a `community_<method>` vertex
   attribute (Leiden-fast / Louvain / label-propagation). When
   `USE_PRECOMPUTED_COMMUNITY` is on and such an attribute is present, we color
   by it directly, so the figure shows *exactly* the communities that notebook
   found.
2. **Leiden, computed here** — for networks without precomputed labels (Test,
   Full, LWCC, 90TS_LWCC, or the un-annotated Final_OutThreshold1), we run
   Leiden (Traag, Waltman & van Eck 2019): the modern successor to Louvain,
   shipped inside igraph, seconds on millions of edges.

Either way, because big social graphs produce hundreds of tiny communities we
relabel them **by size**: the `TOP_COMMUNITIES` largest keep individual colors,
everything else becomes a single grey `"other"` class — this is what keeps the
final figure legible.


In [ ]:
# ---- Community labels: reuse precomputed ones if present, else run Leiden -- #
precomputed = [a for a in G_giant.vs.attributes() if a.lower().startswith("community")]

if USE_PRECOMPUTED_COMMUNITY and precomputed:
    attr = precomputed[0]
    # Map arbitrary stored labels -> contiguous 0..k-1 integers.
    raw = np.asarray(G_giant.vs[attr])
    _uniq, membership = np.unique(raw, return_inverse=True)
    n_comm = int(len(_uniq))
    COMMUNITY_SOURCE = attr.replace("community", "").strip("_") or "precomputed"
    print(f"using precomputed communities from vertex attribute '{attr}': "
          f"{n_comm:,} communities")
else:
    with Stage("Leiden community detection"):
        partition = G_giant.community_leiden(
            objective_function="modularity",   # optimize modularity (CPM is the alternative)
            weights=weights,                   # respect interaction intensity if present
            n_iterations=5,                    # a few refinement passes; cheap and stabler
        )
    membership = np.asarray(partition.membership)
    n_comm = int(membership.max() + 1)
    COMMUNITY_SOURCE = "Leiden"
    print(f"Leiden: {n_comm:,} communities, modularity {partition.modularity:.3f}")

require(n_comm >= 1, "no communities found")

# ---- Relabel communities by size: 0 = largest, 1 = second largest, ... ---- #
sizes = np.bincount(membership)
order = np.argsort(sizes)[::-1]                  # community ids, largest first
rank_of = np.empty_like(order); rank_of[order] = np.arange(n_comm)
comm_rank = rank_of[membership]                  # per-node size rank

# Display label: named color class for the big ones, 'other' for the rest.
comm_display = np.where(comm_rank < TOP_COMMUNITIES,
                        np.char.add("C", comm_rank.astype(str)), "other")

shown = min(TOP_COMMUNITIES, n_comm)
print(f"top {shown} communities cover "
      f"{sizes[order][:shown].sum() / G_giant.vcount():.0%} of nodes")


## 4. Layout on GPU — ForceAtlas2 (classic + linlog)

`cugraph.force_atlas2` runs the real Gephi ForceAtlas2 on the Colab GPU —
**~10–16 s for the full 2M-node graph** (vs ~6 min for CPU grid-FR, and
igraph's DrL never finishes at this scale). Two recipes are rendered:

- **classic** — Gephi defaults (`outbound_attraction_distribution=True`):
  communities separate into distinct regions; hubs and their leaf halos are
  pushed outward instead of sprawling. The social-media map.
- **linlog** — logarithmic attraction: a softer radial-nebula look at the
  default `scaling_ratio` (re-tune scaling if you want LinLog *and* separation).

The install pins cuGraph to the Colab-preinstalled RAPIDS minor version —
unpinned, pip pulls a newer wheel whose `libcugraph.so` is missing.


In [ ]:
# ---- 4.1 GPU check + cuGraph install (version-pinned) ---------------------- #
import shutil, subprocess, sys
HAS_GPU = (shutil.which("nvidia-smi") is not None
           and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0)
print("GPU visible:", HAS_GPU)
if HAS_GPU and not RUNNING_LOCALLY:
    import importlib.metadata as _md
    _pin = ""
    for _anchor in ("cuml-cu12", "cudf-polars-cu12", "cudf-cu12"):
        try:
            _pin = "==" + _md.version(_anchor).rsplit(".", 1)[0] + ".*"
            break
        except _md.PackageNotFoundError:
            continue
    print(f"installing cugraph-cu12{_pin} (matching preinstalled RAPIDS)...")
    !pip install -q "cugraph-cu12{_pin}" "cudf-cu12{_pin}" --extra-index-url=https://pypi.nvidia.com
    # purge any half-imported RAPIDS modules from a previous failed attempt
    for _m in [m for m in sys.modules if m.split(".")[0] in
               ("cugraph", "pylibcugraph", "cudf", "rmm", "pylibcudf")]:
        del sys.modules[_m]
elif not HAS_GPU:
    print("No GPU: Runtime -> Change runtime type -> GPU, then re-run from Setup.")

In [ ]:
# ---- 4.2 Compute both FA2 layouts ------------------------------------------ #
require(HAS_GPU, "a GPU runtime is required (Runtime -> Change runtime type -> "
                 "GPU). CPU fallbacks live in kb:NETWORK_VIZ_STYLES_REF.md")
import cudf, cugraph

def gpu_fa2(g: ig.Graph, params: dict, max_iter: int) -> np.ndarray:
    """cugraph ForceAtlas2 on an igraph Graph; returns (vcount, 2) float array."""
    src, dst = map(np.asarray, zip(*g.get_edgelist()))
    w = (np.asarray(g.es["weight"], dtype="float32")
         if "weight" in g.es.attributes() else np.ones(len(src), dtype="float32"))
    GG = cugraph.Graph()
    GG.from_cudf_edgelist(cudf.DataFrame({"src": src, "dst": dst, "wgt": w}),
                          source="src", destination="dst", weight="wgt",
                          renumber=True)
    pos = cugraph.force_atlas2(GG, max_iter=max_iter,
                               barnes_hut_optimize=True, **params)
    xy = pos.to_pandas().set_index("vertex").sort_index()[["x", "y"]].to_numpy()
    require(xy.shape[0] == g.vcount(), "FA2 returned wrong number of vertices")
    return xy

XY = {}
for _name, _params in FA2_RECIPES.items():
    with Stage(f"FA2 {_name} x{FA2_ITER}"):
        XY[_name] = gpu_fa2(G_giant, _params, FA2_ITER)
print(f"{len(XY)} layouts computed: {list(XY)}")

## 5. The social-media maps (light + dark)

Community-colored ink, no bundling: an edge *inside* a community inherits that
community's color; cross-community edges fade to a neutral `mix` tone. Nodes
are shaded `log` with a high `min_alpha` floor so sparse regions stay
saturated (the floor is per-recipe — the diffuse linlog cloud gets a higher
one). Palettes: `glasbey_dark` on white, `glasbey_light` on near-black.
Positions are saved as parquet, so re-styling never re-runs a layout.


In [ ]:
# ---- 5.1 Style tables: nodes, edges, colored segments ---------------------- #
def normalize_xy(xy: np.ndarray, clip_pct: float = 100.0) -> np.ndarray:
    """Shift/scale coords into [0,1] on the longer axis, keeping aspect.
    clip_pct < 100 first clips each axis to its central window so a handful of
    far-flung outliers can't define the frame."""
    xy = np.asarray(xy, dtype=float)
    if clip_pct < 100.0:
        half = (100.0 - clip_pct) / 2.0
        xy = np.clip(xy, np.percentile(xy, half, axis=0),
                         np.percentile(xy, 100.0 - half, axis=0))
    out = xy - xy.min(axis=0)
    span = out.max(axis=0); span[span == 0] = 1.0
    return out / span.max()

def build_nodes(xy: np.ndarray) -> pd.DataFrame:
    """Canonical node table for one layout (labels/communities from section 3)."""
    xyn = normalize_xy(xy, clip_pct=CLIP_PCT)
    return pd.DataFrame({
        "x": xyn[:, 0], "y": xyn[:, 1],
        "label": G_giant.vs["node_label"],
        "degree": degree,
        "community": pd.Categorical(comm_display),
        "community_rank": comm_rank,
    })

# Edge list is layout-independent - build once.
_src, _tgt = map(np.asarray, zip(*G_giant.get_edgelist()))
edges = pd.DataFrame({
    "source": _src, "target": _tgt,
    "weight": (np.asarray(G_giant.es["weight"], dtype=float)
               if weights else np.ones(len(_src))),
})

def edge_segments(ndf: pd.DataFrame) -> pd.DataFrame:
    """NaN-separated straight segments; intra-community edges keep the
    community label, cross-community edges become 'mix'."""
    cs = ndf["community"].to_numpy()[_src].astype(object)
    ct = ndf["community"].to_numpy()[_tgt].astype(object)
    ecomm = np.where(cs == ct, cs, "mix")
    cats = list(ndf["community"].cat.categories) + ["mix"]
    n = len(_src)
    xs = np.empty(3 * n); ys = np.empty(3 * n)
    xs[0::3] = ndf["x"].to_numpy()[_src]; xs[1::3] = ndf["x"].to_numpy()[_tgt]; xs[2::3] = np.nan
    ys[0::3] = ndf["y"].to_numpy()[_src]; ys[1::3] = ndf["y"].to_numpy()[_tgt]; ys[2::3] = np.nan
    return pd.DataFrame({"x": xs, "y": ys,
                         "community": pd.Categorical(np.repeat(ecomm, 3), categories=cats)})

def color_key(cats, dark=False):
    """Community -> hex; 'other'/'mix' get quiet neutrals per background."""
    pal = cc.glasbey_light if dark else cc.glasbey_dark
    neutral = {"other": "#3a3f4a" if dark else "#d0d0d0",
               "mix":   "#2c3038" if dark else "#dcdcdc"}
    return {c: (neutral[c] if c in neutral else pal[int(c[1:]) % len(pal)])
            for c in cats}

print(f"style helpers ready | {len(edges):,} edges")

In [ ]:
# ---- 5.2 Render: 2 recipes x light/dark = 4 maps --------------------------- #
def render_social_map(ndf, seg, recipe, dark, px=CANVAS_PX):
    mode = "dark" if dark else "light"
    ck  = color_key(ndf["community"].cat.categories, dark)
    eck = color_key(list(ndf["community"].cat.categories) + ["mix"], dark)
    cvs = ds.Canvas(plot_width=px, plot_height=px,
                    x_range=(-0.02, 1.02), y_range=(-0.02, 1.02))
    edge_img = tf.shade(cvs.line(seg, "x", "y", ds.count_cat("community")),
                        color_key=eck, how="eq_hist",
                        alpha=EDGE_ALPHA[mode], min_alpha=8)
    node_img = tf.dynspread(
        tf.shade(cvs.points(ndf, "x", "y", ds.count_cat("community")),
                 color_key=ck, how="log", alpha=255,
                 min_alpha=NODE_MIN_ALPHA[recipe][mode]),
        threshold=0.92, max_px=3)
    img = tf.set_background(tf.stack(edge_img, node_img),
                            DARK_BG if dark else "white")
    arr = np.asarray(img.to_pil().convert("L"))
    frac = (arr > 8).mean() if dark else (arr < 250).mean()
    require(frac > 1e-4, f"social_map_{recipe}_{mode} rendered essentially blank")
    name = f"social_map_{recipe}_{mode}.png"
    img.to_pil().save(OUTPUT_DIR / name)
    print("saved", OUTPUT_DIR / name)
    return img

NDF = {}
for _recipe, _xy in XY.items():
    ndf = build_nodes(_xy)
    with Stage(f"edge segments ({_recipe})"):
        seg = edge_segments(ndf)
    for _dark in (False, True):
        with Stage(f"render {_recipe} {'dark' if _dark else 'light'}"):
            render_social_map(ndf, seg, _recipe, _dark)
    ndf.to_parquet(OUTPUT_DIR / f"positions_{_recipe}.parquet")
    NDF[_recipe] = ndf

nodes = NDF["classic"]        # canonical table for the sections below
nodes.head()

## 6. Coordinate-free views

Past a few million edges *any* node-link picture saturates. Two classic plots
carry structural information at literally any scale, because they never draw
the graph itself.

### 6.1 Degree distribution (CCDF)

The complementary cumulative distribution `P(K ≥ k)` on log-log axes is the
honest way to show a heavy-tailed degree distribution (binning artifacts of
raw histograms disappear). For rigorous tail *fitting*, use the `powerlaw`
package rather than eyeballing a slope.

In [ ]:
deg_sorted = np.sort(degree)
# CCDF: for each observed degree value, the fraction of nodes with degree >= it
ccdf = 1.0 - np.arange(len(deg_sorted)) / len(deg_sorted)

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.loglog(deg_sorted, ccdf, marker=".", markersize=3, linestyle="none",
          color="#3b528b", alpha=0.6)
ax.set_xlabel("degree $k$")
ax.set_ylabel(r"$P(K \geq k)$")
ax.set_title(f"{NETWORK} - degree distribution (CCDF, log-log)")
ax.grid(True, which="both", alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "05_degree_ccdf.png", dpi=200)
plt.show()

### 6.2 Community-blocked adjacency matrix

Order the nodes by community (largest first, by degree inside each), then
2-D-histogram the edge endpoints over that ordering. Dense diagonal blocks =
cohesive communities; off-diagonal smears = the bridges between them. Because
it's a fixed-size histogram, this works unchanged on a 10-million-edge
graph.

In [ ]:
# Permutation: sort nodes by (community size rank, then -degree).
perm = np.lexsort((-nodes["degree"].to_numpy(), nodes["community_rank"].to_numpy()))
pos_of = np.empty(len(perm), dtype=np.int64)
pos_of[perm] = np.arange(len(perm))                    # node id -> matrix row

BINS = 512                                             # fixed cost at any scale
r, c = pos_of[edges["source"].to_numpy()], pos_of[edges["target"].to_numpy()]
H, _, _ = np.histogram2d(np.r_[r, c], np.r_[c, r],     # symmetrize
                         bins=BINS, range=[[0, len(perm)], [0, len(perm)]])

fig, ax = plt.subplots(figsize=(6.5, 6))
im = ax.imshow(np.log1p(H), cmap="magma", origin="upper", interpolation="nearest")
ax.set_title(f"Adjacency matrix, nodes ordered by {COMMUNITY_SOURCE} community")
ax.set_xlabel("node rank"); ax.set_ylabel("node rank")
fig.colorbar(im, ax=ax, label="log(1 + edges per cell)", shrink=0.8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "06_adjacency_blocks.png", dpi=200)
plt.show()

## 7. Save the artifacts

Positions and attributes are the expensive things — persist them so future
sessions (or Gephi) start from here instead of recomputing. The GraphML export
carries `x`/`y`, so opening it in [Gephi](https://gephi.org) /
[Gephi Lite](https://gephi.org/gephi-lite/) shows *this* layout immediately.

In [ ]:
# 1) Node tables (positions + attributes) per recipe - already saved as
#    positions_<recipe>.parquet in section 5. The canonical `nodes` (classic)
#    also feeds the views above.

# 2) Optional GraphML with embedded coordinates & community, for Gephi.
#    ~1.2 GB at 2M nodes -> behind a flag.
if EXPORT_GRAPHML:
    G_export = G_giant.copy()
    G_export.vs["x"] = nodes["x"].tolist()
    G_export.vs["y"] = nodes["y"].tolist()
    G_export.vs["community"] = [str(c) for c in nodes["community"]]
    G_export.vs["degree"] = nodes["degree"].tolist()
    G_export.write_graphml(str(OUTPUT_DIR / "network_with_layout.graphml"))
else:
    print("GraphML export skipped (EXPORT_GRAPHML=False)")

print("Artifacts in", OUTPUT_DIR.resolve())
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {p.name:<40} {p.stat().st_size/1e6:6.1f} MB")

## 8. Run summary

Every stage recorded its wall time, memory and outcome. If a run ever *feels*
slow, this is where you find out which stage to shrink.

In [ ]:
summary = pd.DataFrame(STAGE_LOG)
total = summary["seconds"].sum()
print(f"total tracked time: {total:.0f}s ({total/60:.1f} min)")
summary

## Where to go from here

*Every alternative style* (CPU grid-FR, DrL's measured failure, k-core
reduction, hammer-bundled "rivers", the matplotlib publication composite,
ipysigma interactive HTML, continuous-attribute coloring) lives — with code,
verdicts, and benchmarks — in the knowledge base:
`content/how-to/NETWORK_VISUALIZATION_SKILL.md` and
`content/reference/NETWORK_VIZ_STYLES_REF.md`. *Restyling:* the saved
`positions_<recipe>.parquet` files mean any palette/alpha/background change
re-renders in seconds with no relayout. *Other networks:* switch the `NETWORK`
key in Setup; the same pipeline renders any of the project's GML graphs.
